<a href="https://colab.research.google.com/github/DanielHashmi/Homework_Python_Projects/blob/main/25%20Projects/Stewart_Base_Discord_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import discord
from discord.ext import commands
import os
import google.generativeai as genai
import json
import traceback
import nest_asyncio

intents = discord.Intents.default()
intents.message_content = True
bot = commands.Bot(command_prefix='!', intents=intents)

genai.configure(api_key='gemini_api_key')
model = genai.GenerativeModel('gemini-1.5-flash')

conversation_history = {}
HISTORY_FILE = 'conversation_history.json'

def load_history():
    global conversation_history
    if os.path.exists(HISTORY_FILE):
        try:
            with open(HISTORY_FILE, 'r') as f:
                stored = json.load(f)

            for user_id, history in stored.items():
                if history:
                    chat = model.start_chat()
                    for msg in history:
                        if msg["role"] == "user":
                            chat.send_message(msg["content"])
                    conversation_history[user_id] = chat
        except Exception as e:
            print(f"Error loading history: {e}")
            print(traceback.format_exc())
            if os.path.exists(HISTORY_FILE):
                os.rename(HISTORY_FILE, f"{HISTORY_FILE}.bak")

def save_history():
    try:
        serializable = {}
        for user_id, chat in conversation_history.items():
            history = []
            for msg in chat.history:
                content = msg.parts[0].text if hasattr(msg, 'parts') and msg.parts else ""
                role = msg.role if hasattr(msg, 'role') else "user"
                history.append({"role": role, "content": content})
            serializable[user_id] = history

        with open(HISTORY_FILE, 'w') as f:
            json.dump(serializable, f, indent=2)
    except Exception as e:
        print(f"Error saving history: {e}")
        print(traceback.format_exc())

load_history()

@bot.event
async def on_ready():
    print(f'Logged in as {bot.user.name}')
    print('Stewart Base AI is ready!')
    await bot.change_presence(activity=discord.Activity(
        type=discord.ActivityType.listening,
        name="your questions | Use !ask"
    ))

@bot.command(name="ask")
async def ask(ctx, *, question):
    user_id = str(ctx.author.id)
    async with ctx.typing():
        try:
            if user_id not in conversation_history:
                conversation_history[user_id] = model.start_chat()

            chat = conversation_history[user_id]
            response = chat.send_message(question)
            response_text = response.text

            save_history()

            if len(response_text) <= 1900:
                await ctx.reply(response_text)
            else:
                chunks = [response_text[i:i+1900] for i in range(0, len(response_text), 1900)]
                for i, chunk in enumerate(chunks):
                    if i == 0:
                        await ctx.reply(chunk)
                    else:
                        await ctx.send(chunk)
        except Exception as e:
            await ctx.reply(f"Sorry, I encountered an error: {e}")
            print(traceback.format_exc())

@bot.command(name="reset")
async def reset_conversation(ctx):
    user_id = str(ctx.author.id)
    if user_id in conversation_history:
        conversation_history[user_id] = model.start_chat()
        save_history()
        await ctx.reply("Your conversation with Stewart Base has been reset!")
    else:
        await ctx.reply("You don't have any conversation history yet.")

@bot.command(name="help_stewart")
async def help_command(ctx):
    help_text = """
**Stewart Base AI Bot - Help Guide**
**Commands:**
• `!ask <your question>` - Ask Stewart Base any question
• `!reset` - Reset your conversation history
• `!help_stewart` - Show this help message
    """
    await ctx.send(help_text)

@bot.event
async def on_message(message):
    if message.author.bot:
        return
    await bot.process_commands(message)
    if bot.user.mentioned_in(message) and not message.mention_everyone:
        content = message.content.replace(f'<@{bot.user.id}>', '').strip()
        if content:
            ctx = await bot.get_context(message)
            await ask(ctx, question=content)

if __name__ == "__main__":
    DISCORD_TOKEN = 'discord_token'
    nest_asyncio.apply()
    bot.run(DISCORD_TOKEN)

2025-04-22 04:14:40 INFO     discord.client logging in using static token
2025-04-22 04:14:40 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2025-04-22 04:14:40 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: a3601efeda7c309f46a338cc890f355b).
2025-04-22 04:14:40 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: a3601efeda7c309f46a338cc890f355b).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: a3601efeda7c309f46a338cc890f355b).


Logged in as Stewart Base
Stewart Base AI is ready!
